# Predictions and Risk Interpretation

The prediction products are decision-support surfaces. They are designed to show where species use, environmental support, and fishing exposure overlap in relative terms. They should not be read as absolute bycatch probabilities.

This notebook explains the main prediction layers, how the plausibility gate changes the species-use surface, and how realized, latent, seascape-conditioned, and weekly operator products differ from one another.

## Prediction Product Families

The main prediction products are written as yearly `h3`/`date`/`species` partitions under `data/modeling/predictions/`.

The active hybrid prediction product is:

```text
data/
└── modeling/
    └── predictions/
        └── hybrid_presence_gate_extra_trees_som_hierarchical_k30_5fold_blockcv_bayesian_gmm_k30/
            └── joint/
                └── year=<YYYY>/
                    └── part.parquet
```

The 2022 seascape-conditioned exploratory product is:

```text
data/
└── modeling/
    └── predictions/
        └── seascape_som_15x15_hierarchical_k30/
            └── joint/
                └── year=2022/
                    └── part.parquet
```

The hybrid product keeps both the final gated species-use prediction and intermediate quantities used to interpret it: the raw machine-learning prediction, plausibility, the plausibility gate, transformed fishing exposure, and realized risk.

Example hybrid species-use outputs:

<p>
  <img src="../plots/predictions/hybrid_presence_gate_extra_trees_som_hierarchical_k30_5fold_blockcv_bayesian_gmm_k30_joint_species_use_log_pred_non_zero_mean_BBAL_2022_all_months.png" width="49%" />
  <img src="../plots/predictions/hybrid_presence_gate_extra_trees_som_hierarchical_k30_5fold_blockcv_bayesian_gmm_k30_joint_species_use_log_pred_non_zero_mean_SAFS_2022_all_months.png" width="49%" />
</p>


## Relative Risk Framework

Conceptually, relative risk increases when species use and fishing exposure overlap:

$$
Q(h,t,s)=U(h,t,s)E(h,t)
$$

where $U$ is relative species use and $E$ is fishing exposure for H3 cell $h$, date $t$, and species $s$.

The implemented products are stored on transformed scales. This keeps the surfaces numerically stable and easier to compare, but it also means the values should be interpreted as relative scores, not as raw bycatch rates.

## Environmental Plausibility and Gate

Plausibility is normalized from the Bayesian GMM environmental-support layer:

$$
p_s(h,t)=clip((d_s(h,t)-d_{s,min})/(d_{s,max}-d_{s,min}),0,1)
$$

The plausibility gate is:

$$
g_s(h,t)=1-c_s(1-p_s(h,t))
$$

In this implementation, $c_s = 0.10$, so predictions are damped but not forced to zero under weak environmental support.

A high plausibility score means the `h3`/`date`/`species` feature vector resembles environmental conditions associated with observed positive telemetry use. A low plausibility score does not prove absence; it means the model is operating farther from familiar positive-use conditions.

Example plausibility and gate-sensitivity outputs:

<p>
  <img src="../plots/plausibility/bayesian_gmm_k30_joint_plausibility_non_zero_mean_BBAL_2022.png" width="49%" />
  <img src="../plots/plausibility/bayesian_gmm_k30_joint_plausibility_non_zero_mean_SAFS_2022.png" width="49%" />
</p>

<p>
  <img src="../plots/predictions/hybrid_presence_gate_extra_trees_som_hierarchical_k30_5fold_blockcv_bayesian_gmm_k30_joint_latent_risk_plausibility_gate_sensitivity_2022_small_multiples.png" width="98%" />
</p>


## Hybrid Species Use

The final hybrid species-use layer is the Extra Trees prediction after the plausibility gate has been applied:

$$
U_{hybrid}=log(1+(exp(U_{ML})-1)g)
$$

This expression keeps the Extra Trees model as the main predictor while allowing the plausibility layer to reduce predictions made under weak environmental support. The output remains on the same log scale as the model prediction.

In the yearly prediction partitions, the relevant fields are:

- `species_use_ml_log_pred`: Extra Trees species-use prediction before the plausibility gate.
- `plausibility`: environmental-support score from the Bayesian GMM layer.
- `plausibility_gate`: species-specific gate applied to the prediction.
- `species_use_log_pred`: final hybrid species-use prediction.
- `fishing_activity_log`: transformed fishing exposure.
- `risk_log_pred`: realized risk score on the transformed scale.

## Realized and Latent Risk

Fishing activity is transformed as:

$$
f(h,t)=log(1+F(h,t))
$$

Realized risk is stored additively on the transformed scale:

$$
r_s(h,t)=U^*_{s}(h,t)+f(h,t)
$$

where $U^*_{s}$ is the selected species-use prediction surface. Realized risk therefore depends on where fishing activity actually occurred.

Latent risk uses a standardized baseline exposure of 0.5 vessel-hours per H3 cell-day:

$$
L_s(h,t)=U^*_{s}(h,t)+log(1+0.5)
$$

Latent risk asks where risk could emerge if a small standardized amount of fishing effort were present. This makes it useful for planning and operational awareness, but it is not a record of realized fishing overlap.

Example realized-risk and latent-risk outputs:

<p>
  <img src="../plots/predictions/hybrid_presence_gate_extra_trees_som_hierarchical_k30_5fold_blockcv_bayesian_gmm_k30_joint_risk_log_pred_non_zero_mean_BBAL_2022_all_months.png" width="49%" />
  <img src="../plots/predictions/hybrid_presence_gate_extra_trees_som_hierarchical_k30_5fold_blockcv_bayesian_gmm_k30_joint_risk_log_pred_non_zero_mean_SAFS_2022_all_months.png" width="49%" />
</p>

<p>
  <img src="../plots/predictions/hybrid_presence_gate_extra_trees_som_hierarchical_k30_5fold_blockcv_bayesian_gmm_k30_joint_latent_risk_log_pred_non_zero_mean_BBAL_2022_monthly_matrix.png" width="49%" />
  <img src="../plots/predictions/hybrid_presence_gate_extra_trees_som_hierarchical_k30_5fold_blockcv_bayesian_gmm_k30_joint_latent_risk_log_pred_non_zero_mean_SAFS_2022_monthly_matrix.png" width="49%" />
</p>


## Weekly Operator Products

The weekly operator products summarize latent risk by ISO week. They are derived from the hybrid prediction partitions and are written under:

```text
data/
└── modeling/
    └── weekly_operator/
        └── hybrid_presence_gate_extra_trees_som_hierarchical_k30_5fold_blockcv_bayesian_gmm_k30/
            └── joint/
                ├── latent_risk_iso_week_by_year_2014-2023.parquet
                ├── latent_risk_iso_week_climatology_2014-2023.parquet
                └── latent_risk_iso_week_sequence_2022.parquet
```

These products are important because they move from daily H3 predictions to time windows that are easier to use in operational maps, small multiples, and animations.

Example weekly operator outputs:

<p>
  <img src="../plots/predictions/weekly_operator/hybrid_presence_gate_extra_trees_som_hierarchical_k30_5fold_blockcv_bayesian_gmm_k30_joint_latent_risk_iso_week_climatology_2014-2023_small_multiples.png" width="98%" />
</p>

<p>
  <img src="../plots/predictions/weekly_operator/hybrid_presence_gate_extra_trees_som_hierarchical_k30_5fold_blockcv_bayesian_gmm_k30_joint_latent_risk_iso_week_climatology_2014-2023_fisheries_grid_example.png" width="98%" />
</p>


## Seascape-Conditioned Exploratory Products

The seascape-conditioned products summarize species use and risk by the selected SOM-hierarchical k=30 environmental regimes. They are exploratory: they help inspect how predicted species use and risk vary across recurring environmental states.

The 2022 seascape-conditioned maps use presentation-specific binned scales. Those scales were tuned for interpretability and should be preserved until the visualization module has formal visual regression references.

Example seascape-conditioned species-use and latent-risk matrices:

<p>
  <img src="../plots/predictions/seascape_som_15x15_hierarchical_k30_joint_species_use_log_pred_non_zero_mean_absolute_BBAL_2022_monthly_matrix.png" width="49%" />
  <img src="../plots/predictions/seascape_som_15x15_hierarchical_k30_joint_species_use_log_pred_non_zero_mean_absolute_SAFS_2022_monthly_matrix.png" width="49%" />
</p>

<p>
  <img src="../plots/predictions/seascape_som_15x15_hierarchical_k30_joint_latent_risk_log_pred_non_zero_mean_BBAL_2022_monthly_matrix.png" width="49%" />
  <img src="../plots/predictions/seascape_som_15x15_hierarchical_k30_joint_latent_risk_log_pred_non_zero_mean_SAFS_2022_monthly_matrix.png" width="49%" />
</p>


In [1]:
from pathlib import Path
import pandas as pd
import yaml

config = yaml.safe_load(Path("../config.yaml").read_text())
data_dir = Path("..") / config["paths"]["data"]
plots_dir = Path("..") / config["paths"]["plots"]

model_name = "hybrid_presence_gate_extra_trees_som_hierarchical_k30_5fold_blockcv_bayesian_gmm_k30"
seascape_model = "seascape_som_15x15_hierarchical_k30"

artifacts = {
    "hybrid predictions, 2022": data_dir / "modeling" / "predictions" / model_name / "joint" / "year=2022" / "part.parquet",
    "plausibility, 2022": data_dir / "modeling" / "plausibility" / "bayesian_gmm_k30" / "joint" / "year=2022" / "part.parquet",
    "seascape-conditioned predictions, 2022": data_dir / "modeling" / "predictions" / seascape_model / "joint" / "year=2022" / "part.parquet",
    "weekly latent-risk climatology": data_dir / "modeling" / "weekly_operator" / model_name / "joint" / "latent_risk_iso_week_climatology_2014-2023.parquet",
    "weekly latent-risk sequence, 2022": data_dir / "modeling" / "weekly_operator" / model_name / "joint" / "latent_risk_iso_week_sequence_2022.parquet",
    "prediction plots directory": plots_dir / "predictions",
}

pd.DataFrame(
    [
        {
            "product": name,
            "relative_path": path.relative_to(Path("..")),
            "status": "present" if path.exists() else "missing",
        }
        for name, path in artifacts.items()
    ]
)

,product,relative_path,status
0,"hybrid predictions, 2022",data/modeling/predictions/hybrid_presence_gate...,present
1,"plausibility, 2022",data/modeling/plausibility/bayesian_gmm_k30/jo...,present
2,"seascape-conditioned predictions, 2022",data/modeling/predictions/seascape_som_15x15_h...,present
3,weekly latent-risk climatology,data/modeling/weekly_operator/hybrid_presence_...,present
4,"weekly latent-risk sequence, 2022",data/modeling/weekly_operator/hybrid_presence_...,present
5,prediction plots directory,plots/predictions,present


In [2]:
import pyarrow.parquet as pq

schema_path = artifacts["hybrid predictions, 2022"]
columns = pq.read_schema(schema_path).names
pd.DataFrame({"hybrid_prediction_columns": columns})

,hybrid_prediction_columns
0,h3
1,date
2,species
3,species_use_log_pred
4,hybrid_alpha
5,species_use_ml_log_pred
6,plausibility
7,plausibility_gate
8,fishing_activity_log
9,risk_log_pred


## Plot Groups

The plotting runner can regenerate the main prediction, seascape, weekly, and video products by group:

```bash
python3 scripts/plots/plot_all_maps.py --group predictions
python3 scripts/plots/plot_all_maps.py --group seascapes
python3 scripts/plots/plot_all_maps.py --group weekly
python3 scripts/plots/plot_all_maps.py --group videos
```

The seascape latent-risk monthly matrix uses the retained binned scale through the `seascapes` group. This is one of the cases where plot appearance is part of the interpretation, not just decoration.